# MCP Toolbox DuckDB + Quack — Walkthrough

End-to-end tour of the demo stack. Brings up the services, exercises every toolset (curated queries, metadata discovery, dev-only execute-sql), demonstrates the reconnect path, and points you at Jaeger for distributed tracing.

Run cells top-to-bottom. Shell-flavored cells assume macOS / Linux with `docker compose` available; tool-call cells use plain Python + `requests`.


## Prerequisites

- **Docker** with Compose v2 (`docker compose ...`). macOS users: OrbStack or Docker Desktop both work.
- **The fork checked out at `../mcp-toolbox-duckdb`** — the toolbox image is built from it. If you do not have it yet:
  ```bash
  git clone https://github.com/mitja/mcp-toolbox-duckdb.git ../mcp-toolbox-duckdb
  cd ../mcp-toolbox-duckdb && git checkout feat/duckdb-quack
  ```
- **`ANTHROPIC_API_KEY` in your `.env`** — only needed for the optional LangGraph agent step near the end.
- **Python**: only the `requests` library is needed beyond a standard Jupyter install.

The cell below sets the working directory to the repo root so every subsequent cell can reference paths consistently.


In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("cwd:", os.getcwd())
assert os.path.isfile("docker-compose.yaml"), \
    "Expected to be in the demo repo root with docker-compose.yaml present"


## 1 · Set up `.env`

The compose file expects two env vars:
- `QUACK_TOKEN` — auth token shared between the Quack server and the toolbox source config. Must match `^[A-Za-z0-9_-]{8,}$`.
- `ANTHROPIC_API_KEY` — only needed for the LangGraph agent step (skip the cell if you don't have one).

If `.env` doesn't exist yet, this cell creates it from `.env.example` and rewrites `QUACK_TOKEN`. Edit `.env` afterwards to drop your real `ANTHROPIC_API_KEY` in.


In [ ]:
import os, subprocess

if not os.path.isfile(".env"):
    subprocess.run(["cp", ".env.example", ".env"], check=True)
    # Replace the placeholder QUACK_TOKEN inline
    with open(".env") as f:
        text = f.read()
    text = text.replace("CHANGE-ME", "12345678")
    with open(".env", "w") as f:
        f.write(text)
    print("Created .env from .env.example. "
          "Edit it to set ANTHROPIC_API_KEY before the LangGraph cell.")
else:
    print(".env already exists; leaving alone.")

# Confirm both vars are present (no values printed)
with open(".env") as f:
    keys = {l.split("=", 1)[0] for l in f if "=" in l and not l.startswith("#")}
print("keys in .env:", sorted(keys))


## 2 · Bring up the core stack

`docker compose build` compiles the toolbox image from the sibling fork (slow on first run — Go + CGO + DuckDB native libs, ~5 minutes). After that, `up -d` starts:

| Service | What |
|---|---|
| `quack-server` | Debian + DuckDB CLI + Quack extension loaded; serves the analytics dataset on port 9494 |
| `toolbox` | MCP Toolbox built from the fork; HTTP API on host port 5555, MCP at `/mcp` |
| `otel-collector` | Receives OTLP, prints to stdout, forwards to Jaeger |
| `jaeger` | All-in-one; UI at http://localhost:16686 |


In [ ]:
!docker compose build --quiet quack-server toolbox 2>&1 | tail -3


In [ ]:
!docker compose up -d quack-server toolbox otel-collector jaeger 2>&1 | tail -6


In [ ]:
# Give the containers ~8s to settle, then check status.
import time
time.sleep(8)
!docker compose ps


## 3 · Inspect the served toolsets

`tools.yaml` exposes three toolsets:

- **`analytics_readonly`** — curated query tools (production-shaped)
- **`analytics_metadata`** — discovery tools (production-shaped)
- **`analytics_dev`** — `duckdb-execute-sql` (dev only)

Build a small `invoke()` helper that calls a tool via the HTTP API, then list the toolsets.


In [ ]:
import json, requests

BASE = "http://localhost:5555"


def invoke(tool: str, **args) -> dict:
    """POST /api/tool/<tool>/invoke, unwrap the spec §7 envelope.

    Tools return {"result": "<json-string>"}; this unwraps once.
    Agent-side errors come back inside the envelope and are raised
    here as RuntimeError so cells fail loudly.
    """
    r = requests.post(f"{BASE}/api/tool/{tool}/invoke", json=args, timeout=30)
    r.raise_for_status()
    inner = json.loads(r.json()["result"])
    if isinstance(inner, dict) and "error" in inner:
        raise RuntimeError(inner["error"])
    return inner


def toolset(name: str = "") -> dict:
    path = f"{BASE}/api/toolset/{name}" if name else f"{BASE}/api/toolset"
    return requests.get(path).json()


for name in ["analytics_readonly", "analytics_metadata", "inventory_readonly", "cross_catalog", "analytics_dev"]:
    keys = list(toolset(name).get("tools", {}).keys())
    print(f"{name}: {keys}")


## 4 · Browse tools with MCP Inspector

The [MCP Inspector](https://github.com/modelcontextprotocol/inspector) is a profile-gated service that gives you a browser UI to list, inspect, and invoke each tool — no SDK or LLM required.

Bring up the inspector container, then open the pre-filled URL below in your browser. The query params populate **Transport Type = Streamable HTTP** and **URL = `http://toolbox:5000/mcp`**; just click **Connect**, then **List Tools** in the left pane, pick one, fill in params, and click **Run Tool**.

Notes:
- The URL is `http://toolbox:5000/mcp` (the in-network hostname Toolbox listens on), **not** `http://localhost:5555/mcp`. The Inspector's UI hands the URL off to a proxy process inside its container — `localhost` from there resolves to the inspector itself.
- The Compose service sets `DANGEROUSLY_OMIT_AUTH=true` so the UI is reachable without a session token. Fine for localhost; drop the env var before exposing ports 6274/6277 anywhere else.


In [ ]:
!docker compose --profile inspect up -d inspector 2>&1 | tail -5
print()
print('Open this URL to browse the tools:')
print('  http://localhost:6274/?transport=streamable-http&serverUrl=http%3A%2F%2Ftoolbox%3A5000%2Fmcp')


## 5 · Curated query tools (`analytics_readonly`)

These are the `duckdb-sql` tools. The SQL is fixed in `tools.yaml`; the agent supplies parameter values only.

The response is the spec §7 JSON shape — typed columns, ordered rows, row count, truncation flag, and a statement hash.


In [ ]:
res = invoke("revenue_by_customer", customer_pattern="gmbh")
print(f"row_count={res['row_count']}  truncated={res['truncated']}")
print(f"columns: {[(c['name'], c['type']) for c in res['columns']]}")
for r in res["rows"]:
    print(" ", r)


In [ ]:
res = invoke("top_products")
print(f"row_count={res['row_count']}")
for r in res["rows"]:
    print(" ", r)


## 6 · Metadata tools (`analytics_metadata`)

The discover-the-catalog flow an agent uses before constructing a query:
`list_catalogs` → `list_remote_schemas` → `list_remote_tables` → `describe_<table>` / `summarize_<table>`.

`list_catalogs` runs client-side (against the in-process DuckDB's `information_schema`). The other tools push their SQL to the remote server via Quack's `quack_query()` because attached-catalog metadata is incomplete in DuckDB.


In [ ]:
for tool in ["list_catalogs", "list_remote_schemas", "list_remote_tables"]:
    res = invoke(tool)
    print(f"\n=== {tool} ({res['row_count']} rows) ===")
    for r in res["rows"]:
        print(" ", r)


In [ ]:
for tool in ["describe_sales", "describe_orders"]:
    res = invoke(tool)
    print(f"\n=== {tool} ===")
    for r in res["rows"]:
        print(f"  {r['column_name']:<15} {r['data_type']:<20} nullable={r['is_nullable']}")


In [ ]:
res = invoke("summarize_sales")
print(f"=== summarize_sales ({res['row_count']} rows) ===")
print(f"columns: {[c['name'] for c in res['columns']]}\n")
for r in res["rows"]:
    print(
        f"  {r['column_name']:<12} {r['column_type']:<18} "
        f"min={r['min']:<12} max={r['max']:<12} avg={r['avg']}"
    )


## 7 · Cross-catalog query (multi-attach source)

Most tools wire to a single Quack server, so DuckDB pushes filters and projections down through ATTACH — joins / aggregates / sorts run on the remote side or stream cheaply through the local DuckDB.

The `combined-analytics` source is different: it ATTACHes **both** Quack servers into the same in-process DuckDB via the adapter's `additional_attachments` config. The `product_orders_overview` tool then references both catalogs in one query — DuckDB has to pull rows from each remote and execute the JOIN locally. This is the canonical example of the architectural cost we documented in the README's [Cross-catalog queries and pushdown](../README.md#cross-catalog-queries-and-pushdown) section.

The next cell runs the join, picks a handful of rows, and shows the same shape Jaeger renders for this tool's `duckdb.query` span.


In [ ]:
res = invoke("product_orders_overview")
print(f"{res['row_count']} rows, truncated={res['truncated']}, source={res['source']}")
print()
print(f"columns: {[c['name'] for c in res['columns']]}")
for row in res['rows'][:5]:
    print(" ", row)
print("  ..." if res['row_count'] > 5 else "")


## 8 · Parquet behind a remote Quack (`inventory_readonly`)

Two tools wired to `inventory-quack` exercise a parquet file that sits next to the *remote* DuckDB on `quack-server-2`. The remote DuckDB exposes it as a view in `main` over `read_parquet(...)`; the Toolbox-side in-process DuckDB sees it through the existing `ATTACH` as a normal `inventory_remote.product_price_history` view. No adapter changes — this is the "server-side federation" pattern from [`NOTES.md`](../NOTES.md).

- **`price_history_for_product`** — single streaming scan of the parquet-backed view, filtered by `ILIKE`.
- **`current_prices`** — joins the products table with the open-ended (`valid_to IS NULL`) row in the parquet view. The join is wrapped in `quack_query()` so it runs server-side and only one result stream comes back; without the wrapper, the Toolbox-side DuckDB chokes on two streaming scans in one plan.


In [ ]:
res = invoke("price_history_for_product", product_pattern="Widget")
print(f"{res['row_count']} rows, source={res['source']}")
print(f"columns: {[c['name'] for c in res['columns']]}")
for row in res['rows']:
    print(" ", row)


In [ ]:
res = invoke("current_prices")
print(f"{res['row_count']} rows, source={res['source']}")
print(f"columns: {[c['name'] for c in res['columns']]}")
# Sanity: the current_price values agree with the products
# table's unit_price — the parquet's tail row is the same
# number, just paired with the date it took effect.
for row in res['rows'][:5]:
    print(" ", row)
print("  ..." if res['row_count'] > 5 else "")


## 9 · Semantic-layer integration with Cube — both surfaces

The Cube sidecar holds one set of cube definitions in [`cube/model/cubes/`](../cube/model/cubes); the same model backs the BI surface (Cube REST API on `:4000`, Cube Playground in the browser) and the agent surface (the `analytics_cube_backed` toolset). The `cube_*` tools' SQL is **codegen'd** from the cube YAML — edit a measure's `sql:` fragment, rerun `cube/gen_toolbox_from_cube.py`, the tool's SQL regenerates. See the [`Semantic-layer integration with Cube`](../README.md#semantic-layer-integration-with-cube) section in the README for the architecture story (and why runtime federation via `postgres_scanner` doesn't work today).

Below we hit both surfaces with the same logical query — `{measures: [sales.revenue, sales.order_count], dimensions: [sales.customer]}` — and confirm the rows are identical.

Start Cube first if you haven't already:

```bash
docker compose up -d cube
# Browser UI for the model: http://localhost:4000  (Cube Playground)
```


In [ ]:
# BI surface — hit Cube's REST API directly. Same call any
# Superset/Tableau/etc. connector would make.
import urllib.parse

cube_query = {
    "measures": ["sales.revenue", "sales.order_count"],
    "dimensions": ["sales.customer"],
    "order": [["sales.revenue", "desc"]],
}
qs = urllib.parse.quote(json.dumps(cube_query))
cube_resp = requests.get(f"http://localhost:4000/cubejs-api/v1/load?query={qs}", timeout=15)
cube_resp.raise_for_status()
cube_rows = cube_resp.json()["data"]
print(f"Cube REST  ({len(cube_rows)} rows):")
for r in cube_rows:
    print(" ", r)


In [ ]:
# Agent surface — the corresponding Toolbox tool (codegen'd from
# the same cube YAML; see cube/codegen.yml for the slice config).
tb = invoke("cube_sales_revenue_by_customer")
print(f"Toolbox    ({tb['row_count']} rows, source={tb['source']}):")
for r in tb["rows"]:
    print(" ", r)


In [ ]:
# Side-by-side: both surfaces return the same rows from the same
# semantic model. Numeric formatting differs slightly because
# Cube returns DECIMAL values as bare strings while Toolbox keeps
# the DECIMAL(18,2) precision — normalize before comparing.
from decimal import Decimal

def normalize(row):
    return {
        "customer": row["sales.customer"],
        "revenue": Decimal(str(row["sales.revenue"])),
        "orders": int(row["sales.order_count"]),
    }

cube_norm    = [normalize(r) for r in cube_rows]
toolbox_norm = [normalize(r) for r in tb["rows"]]
assert cube_norm == toolbox_norm, "surfaces disagree — codegen is out of sync"
print(f"\nOK — both surfaces return {len(cube_norm)} identical rows.")
print("Edit cube/model/cubes/sales.yml, rerun cube/gen_toolbox_from_cube.py,")
print("and re-run this section to see the Toolbox SQL follow Cube's model.")


## 10 · BI dashboard with Apache Superset

Same cubes, second surface. Superset connects to Cube's Postgres SQL API (`localhost:15432`) and an idempotent bootstrap registers the Cube database, creates a dataset over the `sales` cube with saved metrics `revenue` (= `MEASURE(revenue)`) and `order_count`, builds a bar chart, and assembles a dashboard. See the [README's BI dashboard with Apache Superset](../README.md#bi-dashboard-with-apache-superset) section for the architecture story (including why Superset works where DuckDB's `postgres_scanner` doesn't).

Start Superset (and Cube, if not already up):

```bash
docker compose up -d cube superset
```

First boot takes ~60s while Superset migrates its metastore. When the bootstrap finishes, the cell below confirms it via Superset's REST API; then open the dashboard URL it prints.


In [ ]:
# Wait for the Superset bootstrap to be reachable + done, then
# print the dashboard URL. The bootstrap is idempotent so re-runs
# of this section are safe.
import time

SUPERSET = "http://localhost:8088"

for _ in range(30):
    try:
        if requests.get(f"{SUPERSET}/health", timeout=3).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)

ss = requests.Session()
r = ss.post(f"{SUPERSET}/api/v1/security/login",
            json={"username":"admin","password":"admin","provider":"db","refresh":True})
r.raise_for_status()
ss.headers["Authorization"] = f"Bearer {r.json()['access_token']}"

# Find the bootstrap'd dashboard (list endpoint), then fetch its
# detail to get the attached charts.
r = ss.get(f"{SUPERSET}/api/v1/dashboard/", params={"q":"(page_size:50)"})
r.raise_for_status()
dashboards = r.json()["result"]
demo = next((d for d in dashboards if d["slug"] == "mcp-toolbox-cube-demo"), None)
assert demo is not None, "dashboard not found — is the bootstrap done?"
detail = ss.get(f"{SUPERSET}/api/v1/dashboard/{demo['id']}").json()["result"]
print(f"dashboard:  {detail['dashboard_title']!r}")
print(f"  charts:   {detail['charts']}")
print(f"  open it:  {SUPERSET}{detail['url']}")
print()
print("Tip: the same six rows you'll see in the bar chart are what")
print("cube_sales_revenue_by_customer returned in §9 — one model,")
print("two surfaces, identical data.")


## 11 · Dev-only tool (`analytics_dev`)

`duckdb-execute-sql` accepts an agent-supplied SQL string. Gated behind `enabled: true` in `tools.yaml` (the toolbox refuses to start otherwise) and logs a `WARN` line every boot.

The **same statement validator** that `duckdb-sql` runs at config-load is applied here **at every invocation**: multi-statement rejected, leading keyword must be in the allowlist, forbidden tokens (DDL/DML/extension/etc.) blocked.


In [ ]:
# Happy path
res = invoke("dev_duckdb_execute_sql",
             sql="SELECT count(*) AS n, max(amount) AS max_amount FROM remote.sales")
print("OK:", res["rows"])


In [ ]:
# Destructive verbs are rejected before reaching the database
for bad_sql in [
    "DROP TABLE remote.sales",
    "INSERT INTO remote.sales VALUES (1, 'x', 1.0, DATE '2026-01-01')",
    "SELECT 1; SELECT 2",
]:
    try:
        invoke("dev_duckdb_execute_sql", sql=bad_sql)
        print(f"unexpectedly accepted: {bad_sql!r}")
    except RuntimeError as e:
        print(f"REJECTED ({bad_sql!r}):\n  {e}\n")


## 12 · Verify the startup warning

The toolbox logs a WARN line every boot when `duckdb-execute-sql` is enabled, so the operator sees it in logs even if they don't read the YAML.


In [ ]:
!docker compose logs toolbox 2>&1 | grep -F 'duckdb-execute-sql is enabled' | head -1


## 13 · Reconnect demo

Restart the Quack server. The toolbox's in-process DuckDB client keeps running, but the next query through the existing TCP conn fails (the server's gone). The source detects this (look for `Invalid connection id` or catalog-missing patterns), pins a fresh `*sql.Conn` from the pool, re-runs `LOAD quack` + `CREATE OR REPLACE SECRET` + `DETACH IF EXISTS` + `ATTACH`, and retries the user query once. The caller never sees the recovery — they just get their rows back.


In [ ]:
!docker compose restart quack-server 2>&1 | tail -3


In [ ]:
# Wait for the new quack-server to become healthy
import subprocess, time
for i in range(1, 16):
    s = subprocess.run(
        ["docker", "inspect", "duckdb-quack", "--format", "{{.State.Health.Status}}"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"  attempt {i}: health={s}")
    if s == "healthy":
        break
    time.sleep(1)


In [ ]:
# Next call should still work — the reconnect path runs transparently
res = invoke("revenue_by_customer", customer_pattern="gmbh")
print(f"row_count={res['row_count']} (recovered after Quack restart)")
for r in res["rows"]:
    print(" ", r)


## 14 · Distributed tracing — Jaeger

Open **http://localhost:16686** in your browser. The OTel collector forwards every span to Jaeger.

To produce a clean cross-service trace, run the `trace-client`: a small OTel-instrumented Python client that puts the W3C `traceparent` in MCP `_meta` so the trace stitches all the way through. (Generic HTTP-level traceparent injection isn't sufficient — Toolbox only extracts trace context from `params._meta.traceparent` in the MCP JSON-RPC body. See [`NOTES.md`](../NOTES.md) for the writeup.)


In [ ]:
!docker compose --profile trace run --rm trace-client 2>&1 | tail -10


Refresh the Jaeger UI now:

1. Pick service `trace-client` from the dropdown.
2. Click "Find Traces".
3. Open the most recent trace.

Expected: **2 services, 5 spans** stitched into one flame graph.

```
trace-client       client.invoke
trace-client         POST            ← HTTP to /mcp (auto-instrumented)
duckdb-quack-demo      toolbox/server/mcp/http
duckdb-quack-demo        tools/call revenue_by_customer
duckdb-quack-demo          duckdb.query   ← our span:
                              db.system = duckdb
                              toolbox.source.name = sales-quack
                              db.statement.parameter_count = 1
                              db.response.rows = 2
                              db.response.truncated = false
```

The `duckdb.query` span also carries a `reattach` event when the reconnect path fires (try running the reconnect cells above with Jaeger open).


## 15 · Optional: LangGraph agent (needs `ANTHROPIC_API_KEY` in `.env`)

A real ReAct agent over the `analytics_readonly` toolset. The agent's reasoning spans (`agent.invoke` + outgoing HTTP spans to Anthropic and Toolbox) land in Jaeger under service `langgraph-demo`.

With `ToolboxClient(..., telemetry_enabled=True)` (the demo passes this), `toolbox-core` injects W3C `traceparent` into MCP `_meta`, and Toolbox extracts it. The whole agent → MCP → Toolbox → quack chain therefore stitches into a single Jaeger trace — confirm by opening any `langgraph-demo` trace after the run completes and seeing `duckdb.query` (service `duckdb-quack-demo`) parented under it.

Skip this cell if you don't have an API key.


In [ ]:
# Build the langgraph image (slow on first run — installs OTel + langchain deps).
!docker compose build --quiet langgraph 2>&1 | tail -3


In [ ]:
!docker compose --profile agent run --rm langgraph 2>&1 | tail -30


## 16 · Connecting from Claude Code

To use the toolset from Claude Code as an MCP client, add Toolbox as an MCP server.

### Per-project (recommended)

Create `.mcp.json` in any project directory:

```json
{
  "mcpServers": {
    "toolbox-analytics": {
      "url": "http://localhost:5555/mcp",
      "transport": "http"
    }
  }
}
```

Next `claude` launch in that directory auto-loads the toolset. Try asking *"What tables can I query?"* — Claude should call `list_remote_tables`, then probably `describe_sales` / `describe_orders`, then formulate an answer.

### Global

Put the same JSON in `~/.claude.json` for repo-independent availability.

### Smoke check the MCP endpoint

If Claude doesn't see the tools, confirm the endpoint is reachable and the toolset is correct via curl:


In [ ]:
import requests

r = requests.post(
    f"{BASE}/mcp",
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
    headers={"Accept": "application/json, text/event-stream"},
    timeout=10,
)
data = r.json()
print(f"Tools advertised via MCP: {len(data['result']['tools'])}\n")
for t in data["result"]["tools"]:
    desc = t["description"].strip().replace("\n", " ")
    print(f"  {t['name']:<25} — {desc[:70]}...")


## 17 · Teardown


In [ ]:
!docker compose down --remove-orphans 2>&1 | tail -3


## Where to go next

- **Real data**: edit `quack-server/seed.sql` to load your own tables (or mount a `.duckdb` file). Update `tools.yaml` with new `duckdb-sql` tools whose statements reference them.
- **Real OTel backend**: replace the `debug` exporter in `otel-collector/config.yaml` with `otlphttp` pointing at Tempo, Honeycomb, Grafana Cloud, etc. The collector config takes effect on `docker compose restart otel-collector`.
- **More tool types**: full tool reference at [`docs/en/integrations/duckdb-quack/tools/`](https://github.com/mitja/mcp-toolbox-duckdb/tree/feat/duckdb-quack/docs/en/integrations/duckdb-quack/tools) in the fork.
- **Known upstream bugs**: four paste-ready writeups in [`NOTES.md`](../NOTES.md) — Quack URI parser, authz-before-ATTACH, `--telemetry-otlp` flag UX, `toolbox-langchain` `_meta.traceparent` gap.
